In [ ]:
import os
import json
import urllib
import pyodbc
import pandas as pd
from sqlalchemy import create_engine


conn = pyodbc.connect('DRIVER={ODBC Driver 18 for SQL Server};SERVER='+server+';DATABASE='+database+';ENCRYPT=yes;UID='+username+';PWD=' + password + ';TrustServerCertificate=YES')
params = urllib.parse.quote_plus('DRIVER={ODBC Driver 18 for SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD=' + password + ';TrustServerCertificate=YES')
engine = create_engine("mssql+pyodbc:///?odbc_connect={}".format(params))

In [ ]:
# Read 2025 controllable costs
costs = pd.read_excel(
    'data/controllable_costs.xlsx',
    sheet_name='1',
    skiprows=5,
    dtype={'Cost': str, 'Cost Element': str}
)[['CRE Cost Category', 'Address', 'Cost Element', 'Cost Element Description', 'Cost', 'Period']]

for i in range(2, 8):
    sheet = pd.read_excel(
        'data/controllable_costs.xlsx',
        sheet_name=str(i),
        skiprows=5,
        dtype={'Cost': str, 'Cost Element': str}
    )[['CRE Cost Category', 'Address', 'Cost Element', 'Cost Element Description', 'Cost', 'Period']]

    costs = pd.concat([costs, sheet], ignore_index=True)

costs['Cost'] = pd.to_numeric(
    costs['Cost'].str.replace(',', '', regex=False).str.replace('$', '', regex=False),
    errors='coerce'
).round(2)

costs.dropna(subset=['Cost'], inplace=True)

costs['Quarter'] = costs['Period'].str[:2]
costs['Year'] = costs['Period'].str[3:].astype(int)

costs.drop('Period', axis=1, inplace=True)
costs.rename(columns={'CRE Cost Category': 'Cost Category'}, inplace=True)

costs['Cost Element'] = pd.to_numeric(costs['Cost Element'], errors='coerce').astype('Int64')

costs.head(3)

In [ ]:
# 2026 Q1 - March YTD
raw = pd.read_excel(
    'data/controllable_costs_q1_2026.xlsx',
    sheet_name='DS Site Detail'
)

q1 = raw.copy(deep=True)
q1['Quarter'] = 'Q1'
q1['Year'] = 2026

q1['G/L Account2'] = q1['G/L Account2'].fillna(q1['Report Level1'])

q1 = q1[
    ['Address', 'CORP FAC Category', 'G/L Account', 'G/L Account2', 'YTD Mar Actual', 'Quarter', 'Year']
].rename(columns={
    'CORP FAC Category': 'Cost Category',
    'YTD Mar Actual': 'Cost',
    'G/L Account': 'Cost Element',
    'G/L Account2': 'Cost Element Description'
}).copy()

q1.dropna(subset=['Cost'], inplace=True)
q1['Cost'] = pd.to_numeric(q1['Cost'], errors='coerce').round(2)
q1.dropna(subset=['Cost'], inplace=True)
q1['Year'] = q1['Year'].astype(int)
q1['Cost Element'] = pd.to_numeric(q1['Cost Element'], errors='coerce').astype('Int64')

q1.head(3)

In [ ]:
# 2026 Q2 - June YTD
raw = pd.read_excel(
    'data/controllable_costs_q2_2026.xlsx',
    sheet_name='DS Site YTD Actuals'
)

q2 = raw.copy(deep=True)
q2['Quarter'] = 'Q2'
q2['Year'] = 2026

q2['G/L Account2'] = q2['G/L Account2'].fillna(q2['Report Level1'])

q2 = q2[
    ['Address', 'CORP FAC Category', 'G/L Account', 'G/L Account2', 'YTD Jun Actual', 'Quarter', 'Year']
].rename(columns={
    'CORP FAC Category': 'Cost Category',
    'YTD Jun Actual': 'Cost',
    'G/L Account': 'Cost Element',
    'G/L Account2': 'Cost Element Description'
}).copy()

q2.dropna(subset=['Cost'], inplace=True)
q2['Cost'] = pd.to_numeric(q2['Cost'], errors='coerce').round(2)
q2.dropna(subset=['Cost'], inplace=True)
q2['Year'] = q2['Year'].astype(int)
q2['Cost Element'] = pd.to_numeric(q2['Cost Element'], errors='coerce').astype('Int64')

# Q2 file is YTD through June, so subtract the matching Q1 March YTD amount.
match_cols = ['Cost Category', 'Address', 'Cost Element', 'Cost Element Description']

q1_totals = (
    q1.groupby(match_cols, dropna=False, as_index=False)['Cost']
      .sum()
      .rename(columns={'Cost': 'Q1 Cost'})
)

q2 = q2.merge(q1_totals, on=match_cols, how='left')
q2['Q1 Cost'] = q2['Q1 Cost'].fillna(0)
q2['Cost'] = (q2['Cost'] - q2['Q1 Cost']).round(2)
q2.drop(columns='Q1 Cost', inplace=True)

q2.head(3)

In [ ]:
# Quick reconciliation check
print('Q1 total:', round(q1['Cost'].sum(), 2))
print('Q2 quarter-only total:', round(q2['Cost'].sum(), 2))
print('2026 H1 total:', round(q1['Cost'].sum() + q2['Cost'].sum(), 2))

In [ ]:
concatted = pd.concat([costs, q1, q2], ignore_index=True)

concatted['Cost Category'] = concatted['Cost Category'].fillna('0 Other')

concatted.to_sql(
    'controllable_costs',
    schema='qmi',
    con=engine,
    if_exists='replace',
    index=False
)